# Phase 4 — the aero-first blade optimization campaign (Colab)

**What this notebook does.** It runs the real optimization: a Bayesian-optimization loop
that proposes blade shapes, simulates each with SU2 to measure the wind it makes
(`J_fan`), and converges toward the best trade of **wind vs. weight vs. deflection** —
folding and feasibility protected as hard constraints. (The earlier
`aero_objective_walkthrough` notebook was just a single-design demo of the objective.)

**Aero-first.** Wind is the goal, so `J_fan` is *maximized*; mass and deflection are
*minimized*; unfoldable / over-mass designs are penalized without wasting a CFD run.

**How it's built** — all logic is tested code in `src/fanopt/`; this notebook only
orchestrates and visualizes:

```
run_phase4_aero.run  ──▶  BladeObjective (vector → J_fan, mass, deflection)
                     └──▶  blade_campaign.run_campaign  (Sobol DoE → qLogNEHVI + TuRBO)
                                                        (checkpoints every iter → resumable)
```

It runs for **hours** and **checkpoints every iteration**, so a disconnected Colab session
resumes where it stopped — just re-run the campaign cell.

*Import note:* the setup cells import stdlib before the repo is on the path; all project +
plotting imports are consolidated in one cell (§3) once the package is installed.

## 1. Get the code

Clone (or sync) the repo and put it on the path. **The redesign code must already be
committed + pushed to `BRANCH`** for this to work.

In [ ]:
import importlib.util, subprocess, sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
BRANCH = "blade-redesign-aero-first"          # branch carrying the aero-first code
REPO_URL = "https://github.com/clingergab/fan-optimization.git"
REPO_DIR = Path("/content/fan-optimization") if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)

for p in (str(REPO_DIR), str(REPO_DIR / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO_DIR)

## 2. Get SU2

Download the SU2 v8.0.1 CPU build (Colab) or find a local one; `SU2_BIN` is passed to the
objective so every CFD eval uses it. (Stdlib + `find_su2` only — the rest imports next.)

In [ ]:
import urllib.request
from fanopt.cfd.phase3 import find_su2

SU2_BIN = find_su2()
if SU2_BIN is None and IN_COLAB:
    SU2_DIR, SU2_VERSION = Path("/content/su2"), "8.0.1"
    if not any(SU2_DIR.rglob("SU2_CFD")):
        url = (f"https://github.com/su2code/SU2/releases/download/"
               f"v{SU2_VERSION}/SU2-v{SU2_VERSION}-linux64.zip")
        print("[su2] downloading", url)
        urllib.request.urlretrieve(url, "/tmp/su2.zip")
        SU2_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", "/tmp/su2.zip", "-d", str(SU2_DIR)], check=True)
    SU2_BIN = str(next(SU2_DIR.rglob("SU2_CFD")))
assert SU2_BIN, "SU2 not found — install SU2_CFD or set SU2_BIN manually"
print("SU2:", SU2_BIN)

## 3. Imports

All project + plotting imports in one cell (now that the package is installed).

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from fanopt.bo.blade_campaign import CampaignConfig, pareto_designs
from fanopt.geometry.blade import BladeParams
from fanopt.geometry.blade_viz import blade_surface_xyz
from run_phase4_aero import run

## 4. Configure the campaign + persistence

**Persistence (why Drive):** the campaign *state* — `checkpoint.npz`, the ledger, and
`pareto.json` — is small and essential, so on Colab we put it on **Google Drive**. That's
what makes the run **resumable**: if Colab disconnects, re-running the campaign cell
reloads the Drive checkpoint and continues instead of restarting. The per-design SU2
*scratch* (thousands of files per eval, regenerable, never reused) stays on local
ephemeral disk so it doesn't bloat Drive.

**Knobs:** `n_init` (Sobol seed designs), `n_iterations` (BO steps), `n_workers` (parallel
SU2 processes ≈ Colab cores), `n_cycles` (plunge cycles per CFD run; ≥5 for a converged
`J_fan`). Values below are a **real run** (~hours) — shrink them for a quick smoke test.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/fanopt/phase4_aero")   # state → Drive (persists)
    SCRATCH_DIR = Path("/content/phase4_scratch")                 # SU2 scratch → ephemeral
else:
    OUT_DIR = REPO_DIR / "data" / "phase4_aero"
    SCRATCH_DIR = None
OUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = CampaignConfig(n_init=16, n_iterations=60, batch_size=1, n_workers=4, seed=0)
OBJ_KW = dict(radial_u=0.5, n_panels=5, n_cycles=5)   # aero-eval fidelity
print("campaign dir (persisted):", OUT_DIR, "| scratch:", SCRATCH_DIR)
print(cfg)

## 5. Run (or resume) the campaign  ⏱️ hours

Runs the Sobol seed, then the BO iterations, with a live bar (best `J_fan` so far).
**Resumable:** it reloads `checkpoint.npz` if present, so re-running after a disconnect
continues rather than restarts.

In [ ]:
state = run(OUT_DIR, su2_bin=SU2_BIN, cfg=cfg, scratch_dir=SCRATCH_DIR, progress=True, **OBJ_KW)
print("done:", state.x.shape[0], "evaluations")

## 6. Results

Load the Pareto set + the per-eval ledger. `J_fan` is on the inflated `MACH=1e-9` scale
(relative only). A Pareto design isn't beaten on *all* of (wind ↑, mass ↓, deflection ↓)
by another.

In [ ]:
pareto = pareto_designs(state)
rows = [json.loads(l) for l in (OUT_DIR / "evaluations.jsonl").read_text().splitlines()]
finite = [r for r in rows if np.isfinite(r["j_fan"])]
print(f"{len(rows)} evals ({len(finite)} feasible) | {len(pareto)} Pareto designs")
if finite:
    print(f"best J_fan = {max(r['j_fan'] for r in finite):.3e}")
for d in sorted(pareto, key=lambda d: d['j_fan'], reverse=True)[:5]:
    print(f"  J_fan={d['j_fan']:.3e}  mass={d['mass_kg']*1000:5.1f} g  "
          f"defl={d['deflection_m']*1e6:6.2f} um  blades={d['params']['blade_count']}")

## 7. Pareto front + convergence

**Left:** every feasible design in the wind–weight plane, Pareto set highlighted — the
menu you choose from. **Right:** best `J_fan` so far vs. evaluation — still improving, or
plateaued?

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.5))
jf = np.array([r["j_fan"] for r in finite]); mg = np.array([r["mass_kg"] for r in finite])*1000
axL.scatter(jf, mg, s=18, c="lightgray", label="all feasible")
axL.scatter([d["j_fan"] for d in pareto], [d["mass_kg"]*1000 for d in pareto],
            s=45, c="crimson", label="Pareto")
axL.set_xlabel("J_fan (wind, relative)"); axL.set_ylabel("mass (g)")
axL.set_title("Wind vs weight"); axL.legend(); axL.grid(alpha=0.3)

order = np.argsort([r["iteration"] for r in finite])
axR.plot(np.maximum.accumulate(jf[order]), lw=1.8)
axR.set_xlabel("feasible evaluation"); axR.set_ylabel("best J_fan so far")
axR.set_title("Convergence"); axR.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8. The top 3 blades, in 3D

The three highest-`J_fan` Pareto blades, rendered from their parameters (top face =
viridis, bottom = grey). Rotate them — these are the shapes the optimizer *found*: the
dish, the camber/zigzag the displacement grid discovered, and the thicker rib edges.

In [ ]:
top3 = sorted(pareto, key=lambda d: d["j_fan"], reverse=True)[:3]
fig = make_subplots(
    rows=1, cols=len(top3), specs=[[{"type": "surface"}] * len(top3)],
    subplot_titles=[f"J_fan={d['j_fan']:.2e}<br>{d['mass_kg']*1000:.0f} g / "
                    f"{d['params']['blade_count']} blades" for d in top3])
for k, d in enumerate(top3, start=1):
    p = BladeParams.from_dict(d["params"])
    for face, cs in (("top", "Viridis"), ("bottom", "Greys")):
        X, Y, Z = blade_surface_xyz(p, n_radial=55, n_tangential=40, face=face)
        fig.add_trace(go.Surface(x=X*1000, y=Y*1000, z=Z*1000, colorscale=cs,
                                 showscale=False, opacity=0.96), row=1, col=k)
fig.update_layout(height=460, title_text="Top-3 blades by J_fan (mm)", margin=dict(t=90))
fig.show()

## 9. Reading it + what's next

- **The Pareto front is the menu:** pick the wind/weight balance you want, then print those
  blades for the Phase-6 feel test.
- **If convergence has plateaued**, the optimizer has found its best *within this
  parameterization* — the signal it's time for the V2 free-form search (vents, arbitrary
  structure).

**Caveats:** `J_fan` is relative (`MACH=1e-9` scale); this is single mid-radius 2D-slice
screening fidelity — the top picks should get a 3D verification pass (Phase 5) before the
final print; structure here is a coarse mass + analytic-deflection proxy (fine structural
optimization is V1.5).